# Experiment 5a: Leakage audit and near-duplicate-aware (grouped) split

**Why.** The Kaggle Alzheimer MRI dataset has no patient IDs and our split is made image by image, so slices of the same subject can sit in both train and test.
This notebook (1) *measures* how similar the original test images are to training images, (2) builds a **grouped split** where near-identical slices are kept together,
and (3) saves the grouped split plus the audit numbers. It needs **no GPU** (set Accelerator = None; runtime a few minutes).

Method: 64x64 grayscale thumbnails (dataset-mean image removed) -> correlation similarity -> images linked when correlation > tau -> connected components = near-duplicate groups -> whole groups assigned to train/val/test (about 70/15/15, per-class balanced).
The rule for tau is fixed *before* looking at any accuracy: the lowest tau on a preset ladder for which no group holds more than 5% of the images (2% and 10% are reported as sensitivity checks).

**Kaggle setup:** same dataset added as input, Accelerator None, Internet not required. Save Version -> Save & Run All (Commit).

In [ ]:
# 1. Setup and file index (same logic as Experiments 1-4)
import os, json
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
DATA_ROOT = os.environ.get("DATA_ROOT", "/kaggle/input")
SPLIT_SEED = 42; MAIN_FRAC = 0.05; SENS_FRACS = [0.02, 0.10]   # rule fixed in advance: largest near-duplicate group may hold at most 5% of all images
CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
def normalize(n): return n.lower().replace("_", "").replace("-", "").replace(" ", "")
canon = {normalize(c): c for c in CLASSES}
class_dirs = {}
for root, dirs, files in os.walk(DATA_ROOT):
    nm = normalize(os.path.basename(root))
    if nm in canon and any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
        class_dirs.setdefault(canon[nm], []).append(root)
assert all(c in class_dirs for c in CLASSES), "Could not find all 4 class folders."
records = []
for cls, ds in class_dirs.items():
    for d_ in ds:
        for f in os.listdir(d_):
            if f.lower().endswith((".jpg", ".jpeg", ".png")): records.append({"filepath": os.path.join(d_, f), "label": cls})
df = pd.DataFrame(records).drop_duplicates(subset="filepath").reset_index(drop=True)
print("Total images:", len(df)); print(df["label"].value_counts().to_string())


In [ ]:
# --- Near-duplicate-aware (grouped) split: deterministic, CPU-only, no patient IDs needed ---
from PIL import Image

def pixel_features(paths, side=64):
    """64x64 grayscale thumbnails; per-image mean removed, then the dataset-mean image removed (so the shared head/brain template
    does not dominate), then L2-normalised: a dot product is a correlation, 1.0 = identical up to brightness/contrast."""
    X = np.empty((len(paths), side * side), dtype=np.float64)
    for i, p in enumerate(paths):
        im = Image.open(p).convert("L").resize((side, side), Image.BOX)
        X[i] = np.asarray(im, dtype=np.float64).ravel()
    X -= X.mean(axis=1, keepdims=True)
    X -= X.mean(axis=0, keepdims=True)
    X /= (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    return X

def threshold_groups(X, max_frac=0.05, ladder=(0.999, 0.995, 0.99, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50)):
    """Near-duplicate groups = connected components of the graph 'correlation > tau'. tau is chosen by a rule fixed in advance:
    the LOWEST tau on the ladder whose largest component holds at most max_frac of all images (lower tau = stricter separation)."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components
    n = len(X); S = np.round(X @ X.T, 6); np.fill_diagonal(S, -1.0)
    table = []; chosen = None
    for tau in ladder:
        ncomp, lab = connected_components(csr_matrix(S > tau), directed=False)
        sizes = np.bincount(lab)
        table.append({"tau": tau, "n_groups": int(ncomp), "largest_group": int(sizes.max()), "singletons": int((sizes == 1).sum())})
        if sizes.max() <= max_frac * n: chosen = (tau, lab)
    if chosen is None: chosen = (ladder[0], connected_components(csr_matrix(S > ladder[0]), directed=False)[1])
    return chosen[1], chosen[0], table

def assign_groups(groups, labels, n_cls=4, fractions=(0.70, 0.15, 0.15)):
    """Greedy, deterministic assignment of whole groups to train/val/test that keeps per-class proportions close to 70/15/15.
    Groups containing rare classes are placed first."""
    tot = np.bincount(labels, minlength=n_cls).astype(float)
    target = np.array([tot * f for f in fractions])
    cur = np.zeros_like(target)
    gids = np.unique(groups)
    comp = {g: np.bincount(labels[groups == g], minlength=n_cls).astype(float) for g in gids}
    order = sorted(gids, key=lambda g: (-comp[g][3], -comp[g][2], -comp[g].sum(), g))
    split_of = {}
    for g in order:
        scores = [float((comp[g] * (cur[s] / np.maximum(target[s], 1.0))).sum()) for s in range(3)]
        s = int(np.argmin(scores)); split_of[g] = s; cur[s] += comp[g]
    # local search: move single groups between splits while the relative per-class deviation from the 70/15/15 targets shrinks
    def cost(c): return float((((c - target) / np.maximum(target, 1.0)) ** 2).sum())
    improved = True
    while improved:
        improved = False
        for g in order:
            s0 = split_of[g]; best = (cost(cur), s0)
            for s1 in range(3):
                if s1 == s0: continue
                trial = cur.copy(); trial[s0] -= comp[g]; trial[s1] += comp[g]
                cst = cost(trial)
                if cst < best[0] - 1e-12: best = (cst, s1)
            if best[1] != s0:
                cur[s0] -= comp[g]; cur[best[1]] += comp[g]; split_of[g] = best[1]; improved = True
    return np.array([split_of[g] for g in groups])

def leakage_report(X, y, tr, te, name):
    """For every test image: correlation with its most similar TRAIN image, and 1-nearest-neighbour accuracy against train."""
    S = X[te] @ X[tr].T
    nn = S.argmax(axis=1); mx = S.max(axis=1)
    acc = float((y[tr][nn] == y[te]).mean())
    r = {"split": name, "p10_max_corr": float(np.percentile(mx, 10)), "median_max_corr": float(np.median(mx)), "p90_max_corr": float(np.percentile(mx, 90)),
         "frac_gt_0.99": float((mx > 0.99).mean()), "frac_gt_0.95": float((mx > 0.95).mean()), "frac_gt_0.90": float((mx > 0.90).mean()), "nn1_accuracy": acc}
    print(f"{name:36s} corr p10/med/p90 {r['p10_max_corr']:.3f}/{r['median_max_corr']:.3f}/{r['p90_max_corr']:.3f} | frac>0.99 {r['frac_gt_0.99']:.3f} >0.95 {r['frac_gt_0.95']:.3f} >0.90 {r['frac_gt_0.90']:.3f} | 1-NN acc {100*acc:.2f}%")
    return r


In [ ]:
# 2. Reproduce the ORIGINAL split exactly (same as Experiments 1-4) and audit it
y_all = np.array([label_to_idx[l] for l in df["label"]])
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SPLIT_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SPLIT_SEED)
print("Original split:", len(train_df), len(val_df), len(test_df))
path_to_i = {p: i for i, p in enumerate(df["filepath"])}
tr0 = np.array([path_to_i[p] for p in train_df["filepath"]]); va0 = np.array([path_to_i[p] for p in val_df["filepath"]]); te0 = np.array([path_to_i[p] for p in test_df["filepath"]])
X = pixel_features(df["filepath"].tolist())
audit = []
audit.append(leakage_report(X, y_all, tr0, te0, "ORIGINAL split: test -> train"))
audit.append(leakage_report(X, y_all, tr0, va0, "ORIGINAL split: val -> train"))
# reference: how similar are images of DIFFERENT classes? (shows what 'unrelated' looks like)
rng = np.random.default_rng(0); perm = rng.permutation(len(df))
audit.append(leakage_report(X, y_all, perm[:len(tr0)], perm[len(tr0):len(tr0)+len(te0)], "RANDOM split (another draw)"))


In [ ]:
# 3. Near-duplicate groups (connected components of 'correlation > tau'), tau chosen by the pre-set rule; audit each variant
order_idx = np.argsort(df["filepath"].values)           # deterministic order independent of os.listdir
dfs = df.iloc[order_idx].reset_index(drop=True); Xs = X[order_idx]; ys = y_all[order_idx]
results = {}; ladder_table = None
for frac in [MAIN_FRAC] + SENS_FRACS:
    groups, tau, table = threshold_groups(Xs, max_frac=frac)
    if ladder_table is None:
        ladder_table = table
        print("Ladder (tau -> number of groups, largest group, singletons):")
        for t in table: print("  ", t)
    split = assign_groups(groups, ys)
    sizes = np.bincount(groups)
    mixed = np.mean([len(np.unique(ys[groups == g])) > 1 for g in np.unique(groups)])
    tr, va, te = [np.flatnonzero(split == s) for s in range(3)]
    print(f"\nmax group fraction {frac}: chosen tau={tau} | {len(sizes)} groups, median size {np.median(sizes):.0f}, largest {sizes.max()}, groups with >1 class {100*mixed:.1f}% | train/val/test = {len(tr)}/{len(va)}/{len(te)}")
    for nm, ix in [("train", tr), ("val", va), ("test", te)]:
        print("   ", nm, np.bincount(ys[ix], minlength=4).tolist())
    audit.append(leakage_report(Xs, ys, tr, te, f"GROUPED frac={frac} tau={tau}: test->train"))
    audit.append(leakage_report(Xs, ys, tr, va, f"GROUPED frac={frac} tau={tau}: val->train"))
    results[frac] = (groups, split, tau)
groups, split, tau_main = results[MAIN_FRAC]
out = dfs.assign(group=groups, split=np.array(["train", "val", "test"])[split])
out.to_csv("grouped_split.csv", index=False)
pd.DataFrame(audit).to_csv("exp5_leakage_audit.csv", index=False)
pd.DataFrame(ladder_table).to_csv("exp5_tau_ladder.csv", index=False)
print("\nMAIN tau =", tau_main, "| Saved grouped_split.csv, exp5_leakage_audit.csv, exp5_tau_ladder.csv")
print(pd.DataFrame(audit).round(3).to_string(index=False))


## What to download and send back
`exp5_leakage_audit.csv`, `grouped_split.csv`, and the printed log (copy the cell output text). 

**How to read it.** If the original test images have a median max-correlation to training images near 0.95+ and a 1-NN accuracy that is already very high, leakage is plausible and large.
The grouped split should show clearly lower similarity. Accuracy under the grouped split (next notebooks) is then the more honest estimate.